In [28]:
import os
import ast
import pandas as pd
import urllib.request
import re
import spacy
import pickle
import pandas as pd
import time
import json

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from scipy.sparse import hstack, csr_matrix

In [2]:
path = '/kaggle/input/datasets/timospinde/babe-media-bias-annotations-by-experts/data/final_labels_SG1.xlsx'
df = pd.read_excel(path)

print(df.shape)
print(df.columns.tolist())
df.head(10)

(1700, 8)
['text', 'news_link', 'outlet', 'topic', 'type', 'label_bias', 'label_opinion', 'biased_words']


,text,news_link,outlet,topic,type,label_bias,label_opinion,biased_words
0,The Republican president assumed he was helpin...,http://www.msnbc.com/rachel-maddow-show/auto-i...,msnbc,environment,left,Biased,Expresses writer’s opinion,[]
1,Though the indictment of a woman for her own p...,https://eu.usatoday.com/story/news/nation/2019...,usa-today,abortion,center,Non-biased,Somewhat factual but also opinionated,[]
2,Ingraham began the exchange by noting American...,https://www.breitbart.com/economy/2020/01/12/d...,breitbart,immigration,right,No agreement,No agreement,['flood']
3,The tragedy of America’s 18 years in Afghanist...,http://feedproxy.google.com/~r/breitbart/~3/ER...,breitbart,international-politics-and-world-news,right,Biased,Somewhat factual but also opinionated,"['tragedy', 'stubborn']"
4,The justices threw out a challenge from gun ri...,https://www.huffpost.com/entry/supreme-court-g...,msnbc,gun-control,left,Non-biased,Entirely factual,[]
5,A review of his posts in online message boards...,https://eu.usatoday.com/story/news/nation/2020...,usa-today,white-nationalism,center,Biased,Entirely factual,['plant']
6,After the airstrike that killed Iranian Gen. Q...,https://eu.usatoday.com/story/news/politics/20...,usa-today,immigration,center,Non-biased,Entirely factual,[]
7,Investigators believe parents would use falsif...,https://eu.usatoday.com/story/news/nation/2020...,usa-today,vaccines,center,Non-biased,Entirely factual,[]
8,Politicians have no business directing or defi...,https://thefederalist.com/2017/01/24/trumps-na...,federalist,white-nationalism,right,Biased,Expresses writer’s opinion,['sloganeering']
9,"Despite repeated setbacks, Smart made history ...",https://www.nbcnews.com/news/nbcblk/stay-close...,msnbc,sport,left,Non-biased,Entirely factual,[]


In [3]:

# Drop rows with no annotator agreement on the main bias label
df_clean = df[df['label_bias'] != 'No agreement'].copy()

# Parse biased_words from string to actual list
def parse_words(x):
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return []

df_clean['biased_words_list'] = df_clean['biased_words'].apply(parse_words)

# Binary target: 1 = Biased, 0 = Non-biased
df_clean['label'] = (df_clean['label_bias'] == 'Biased').astype(int)

print(df_clean.shape)
print(df_clean['label'].value_counts())
print()
# sanity check a few parsed rows
for i in range(3):
    print(df_clean['text'].iloc[i][:80])
    print(df_clean['biased_words_list'].iloc[i])
    print('---')

(1546, 10)
label
0    800
1    746
Name: count, dtype: int64

The Republican president assumed he was helping the industry at the expense of t
[]
---
Though the indictment of a woman for her own pregnancy loss is unusual in Alabam
[]
---
The tragedy of America’s 18 years in Afghanistan has been a stubborn refusal to 
['tragedy', 'stubborn']
---


In [4]:

X = df_clean['text']
y = df_clean['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words='english'
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_train_tfidf, y_train)

y_pred = clf.predict(X_test_tfidf)

print(classification_report(y_test, y_pred, target_names=['Non-biased', 'Biased']))
print()
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

  Non-biased       0.67      0.63      0.65       160
      Biased       0.63      0.67      0.65       150

    accuracy                           0.65       310
   macro avg       0.65      0.65      0.65       310
weighted avg       0.65      0.65      0.65       310


Confusion matrix:
[[101  59]
 [ 49 101]]


In [5]:
url = "https://raw.githubusercontent.com/ms8r/mpqa/master/subjclues.tff"
urllib.request.urlretrieve(url, "/kaggle/working/subjclues.tff")

with open("/kaggle/working/subjclues.tff") as f:
    for _ in range(5):
        print(f.readline().strip())

type=weaksubj len=1 word1=abandoned pos1=adj stemmed1=n priorpolarity=negative
type=weaksubj len=1 word1=abandonment pos1=noun stemmed1=n priorpolarity=negative
type=weaksubj len=1 word1=abandon pos1=verb stemmed1=y priorpolarity=negative
type=strongsubj len=1 word1=abase pos1=verb stemmed1=y priorpolarity=negative
type=strongsubj len=1 word1=abasement pos1=anypos stemmed1=y priorpolarity=negative


In [6]:
def parse_mpqa(path):
    lexicon = {}
    with open(path) as f:
        for line in f:
            fields = dict(item.split('=') for item in line.strip().split(' ') if '=' in item)
            word = fields.get('word1')
            strength = fields.get('type')          # strongsubj / weaksubj
            polarity = fields.get('priorpolarity')  # positive / negative / neutral / both
            if word:
                lexicon[word] = (strength, polarity)
    return lexicon

mpqa_lex = parse_mpqa("/kaggle/working/subjclues.tff")
print(len(mpqa_lex), "words loaded")

# sanity check a few known loaded words
for w in ['devastating', 'admitted', 'forced', 'tragedy', 'stubborn']:
    print(w, '->', mpqa_lex.get(w, 'NOT FOUND'))

6886 words loaded
devastating -> ('strongsubj', 'negative')
admitted -> NOT FOUND
forced -> NOT FOUND
tragedy -> ('weaksubj', 'negative')
stubborn -> ('strongsubj', 'negative')


In [7]:

def mpqa_features(text, lexicon):
    words = re.findall(r"[a-zA-Z']+", text.lower())
    total = len(words) if words else 1

    strongsubj = sum(1 for w in words if lexicon.get(w, (None, None))[0] == 'strongsubj')
    weaksubj = sum(1 for w in words if lexicon.get(w, (None, None))[0] == 'weaksubj')
    negative = sum(1 for w in words if lexicon.get(w, (None, None))[1] == 'negative')
    positive = sum(1 for w in words if lexicon.get(w, (None, None))[1] == 'positive')

    return {
        'strongsubj_ratio': strongsubj / total,
        'weaksubj_ratio': weaksubj / total,
        'negative_ratio': negative / total,
        'positive_ratio': positive / total,
        'subj_word_count': strongsubj + weaksubj,
    }

feat_df = df_clean['text'].apply(lambda t: mpqa_features(t, mpqa_lex)).apply(pd.Series)
df_clean = pd.concat([df_clean.reset_index(drop=True), feat_df.reset_index(drop=True)], axis=1)

print(df_clean[['text', 'label', 'strongsubj_ratio', 'weaksubj_ratio', 'negative_ratio', 'positive_ratio']].head(10))
print()


                                                text  label  strongsubj_ratio  \
0  The Republican president assumed he was helpin...      1          0.103448   
1  Though the indictment of a woman for her own p...      0          0.031250   
2  The tragedy of America’s 18 years in Afghanist...      1          0.060000   
3  The justices threw out a challenge from gun ri...      0          0.000000   
4  A review of his posts in online message boards...      1          0.043478   
5  After the airstrike that killed Iranian Gen. Q...      0          0.000000   
6  Investigators believe parents would use falsif...      0          0.038462   
7  Politicians have no business directing or defi...      1          0.111111   
8  Despite repeated setbacks, Smart made history ...      0          0.060606   
9  Last week, Ankara broke the 2015 agreement wit...      1          0.000000   

   weaksubj_ratio  negative_ratio  positive_ratio  
0        0.000000        0.000000        0.068966  
1   

In [8]:
df_clean.groupby('label')[['strongsubj_ratio', 'weaksubj_ratio', 'subj_word_count']].mean()

,strongsubj_ratio,weaksubj_ratio,subj_word_count
label,,,
0,0.044733,0.078411,4.077500
1,0.069259,0.086734,5.489276


In [9]:
feature_cols = ['strongsubj_ratio', 'weaksubj_ratio', 'negative_ratio', 'positive_ratio', 'subj_word_count']

X_text = df_clean['text']
X_mpqa = df_clean[feature_cols].values
y = df_clean['label']

# split indices so text and MPQA features stay aligned
idx_train, idx_test = train_test_split(
    df_clean.index, test_size=0.2, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_text.loc[idx_train])
X_test_tfidf = vectorizer.transform(X_text.loc[idx_test])

X_train_combined = hstack([X_train_tfidf, csr_matrix(df_clean.loc[idx_train, feature_cols].values)])
X_test_combined = hstack([X_test_tfidf, csr_matrix(df_clean.loc[idx_test, feature_cols].values)])

y_train, y_test = y.loc[idx_train], y.loc[idx_test]

clf2 = LogisticRegression(max_iter=1000, class_weight='balanced')
clf2.fit(X_train_combined, y_train)

y_pred2 = clf2.predict(X_test_combined)

print(classification_report(y_test, y_pred2, target_names=['Non-biased', 'Biased']))
print()
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred2))

              precision    recall  f1-score   support

  Non-biased       0.68      0.64      0.66       160
      Biased       0.64      0.68      0.66       150

    accuracy                           0.66       310
   macro avg       0.66      0.66      0.66       310
weighted avg       0.66      0.66      0.66       310


Confusion matrix:
[[102  58]
 [ 48 102]]


In [10]:
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

models = {
    'Logistic Regression (baseline)': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Linear SVM': LinearSVC(class_weight='balanced', max_iter=5000),
    'Random Forest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42),
}

results = {}

for name, model in models.items():
    model.fit(X_train_combined, y_train)
    preds = model.predict(X_test_combined)
    report = classification_report(y_test, preds, target_names=['Non-biased', 'Biased'], output_dict=True)
    results[name] = report
    print(f"=== {name} ===")
    print(classification_report(y_test, preds, target_names=['Non-biased', 'Biased']))
    print()

=== Logistic Regression (baseline) ===
              precision    recall  f1-score   support

  Non-biased       0.68      0.64      0.66       160
      Biased       0.64      0.68      0.66       150

    accuracy                           0.66       310
   macro avg       0.66      0.66      0.66       310
weighted avg       0.66      0.66      0.66       310


=== Linear SVM ===
              precision    recall  f1-score   support

  Non-biased       0.67      0.64      0.65       160
      Biased       0.63      0.66      0.64       150

    accuracy                           0.65       310
   macro avg       0.65      0.65      0.65       310
weighted avg       0.65      0.65      0.65       310


=== Random Forest ===
              precision    recall  f1-score   support

  Non-biased       0.63      0.62      0.63       160
      Biased       0.60      0.62      0.61       150

    accuracy                           0.62       310
   macro avg       0.62      0.62      0.62   

In [11]:
summary = pd.DataFrame({
    name: {
        'accuracy': r['accuracy'],
        'biased_precision': r['Biased']['precision'],
        'biased_recall': r['Biased']['recall'],
        'biased_f1': r['Biased']['f1-score'],
        'macro_f1': r['macro avg']['f1-score'],
    }
    for name, r in results.items()
}).T

summary.round(3)

,accuracy,biased_precision,biased_recall,biased_f1,macro_f1
Logistic Regression (baseline),0.658,0.638,0.680,0.658,0.658
Linear SVM,0.648,0.631,0.660,0.645,0.648
Random Forest,0.619,0.604,0.620,0.612,0.619
XGBoost,0.623,0.605,0.633,0.619,0.623


In [12]:

with open('/kaggle/working/bias_classifier.pkl', 'wb') as f:
    pickle.dump(clf2, f)

with open('/kaggle/working/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

with open('/kaggle/working/mpqa_lexicon.pkl', 'wb') as f:
    pickle.dump(mpqa_lex, f)

print("Saved classifier, vectorizer, and lexicon.")


Saved classifier, vectorizer, and lexicon.


In [13]:
nlp = spacy.load("en_core_web_sm")

In [14]:
labeled = pd.read_csv('/kaggle/input/datasets/hannahigboke/sentences-labeled/sentences_labeled.csv')

In [16]:
!pip install llama-cpp-python --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 25.8 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.5 MB/s eta 0:00:00


In [18]:
!pip install huggingface_hub --quiet

from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="Qwen/Qwen2.5-0.5B-Instruct-GGUF",
    filename="qwen2.5-0.5b-instruct-q4_k_m.gguf"
)
print(model_path)

qwen2.5-0.5b-instruct-q4_k_m.gguf:   0%|          | 0.00/491M [00:00<?, ?B/s]

/root/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B-Instruct-GGUF/snapshots/9217f5db79a29953eb74d5343926648285ec7e67/qwen2.5-0.5b-instruct-q4_k_m.gguf


In [19]:
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    n_ctx=2048,
    n_threads=4,
    verbose=False
)

response = llm.create_chat_completion(
    messages=[{"role": "user", "content": "Say hello in one short sentence."}],
    max_tokens=50
)
print(response['choices'][0]['message']['content'])

Hello! It's a pleasure to meet you!


In [20]:
labeled = pd.read_csv('/kaggle/input/datasets/hannahigboke/sentences-labeled/sentences_labeled.csv')
print(labeled.shape)
labeled.head()

(40, 10)


,index,topic,text,emotional_amplification,weasel_attribution,certainty_distortion,implicit_judgment,selective_emphasis,dehumanising_glorifying_framing,general_subjective_language
0,0,gender,"For Fox, Clinton, Omar, and Ocasio-Cortez repr...",0,0,0,0,1,0,0
1,1,student-debt,"At the time, Moynihan was primarily concerned ...",1,0,0,0,0,0,0
2,2,trump-presidency,Trump proceeded to ignore multiple other attem...,0,0,0,1,0,0,0
3,3,elections-2020,Never before have the freewheeling inclination...,0,0,1,0,0,0,0
4,4,abortion,Father Horan was one of the many voices from t...,1,0,0,0,0,1,0


In [21]:
dehumanising_words = {
    "wasteland", "infestation", "flood", "swarm", "invasion", "plague",
    "vermin", "cockroaches", "cancer", "disease", "barren", "dictator",
    "regime", "thugs", "mob"
}
glorifying_words = {
    "hero", "heroic", "champion", "savior", "visionary", "trailblazer",
    "legend", "icon", "crusader", "warrior"
}

def check_dehumanising_glorifying(text):
    words_lower = [t.lower() for t in text.split()]
    dehuman_hits = [w for w in words_lower if w.strip('.,!?"\'') in dehumanising_words]
    glorify_hits = [w for w in words_lower if w.strip('.,!?"\'') in glorifying_words]
    if dehuman_hits or glorify_hits:
        return True, dehuman_hits + glorify_hits
    return False, []

# quick test
for t in [df_clean['text'].iloc[6], df_clean['text'].iloc[11]]:
    print(t)
    print(check_dehumanising_glorifying(t))
    print('---')

Investigators believe parents would use falsified records so their children could attend schools that would otherwise require all students to be vaccinated, Dart told WBBM-TV.
(False, [])
---
Ohio's Down syndrome abortion ban and similar proposals around the country have triggered emotional debate over women’s rights, parental love, and the trust between doctor and patient.
(False, [])
---


In [23]:
import re

selective_emphasis_signals = {
    "every", "all", "none", "only", "solely", "entirely", "completely",
    "underscores", "reaffirms"
}
selective_emphasis_phrases = {
    "the fact that", "proves that"
}

def check_selective_emphasis(text):
    text_lower = text.lower()
    hits = [w for w in selective_emphasis_signals if re.search(r'\b' + re.escape(w) + r'\b', text_lower)]
    hits += [p for p in selective_emphasis_phrases if p in text_lower]
    return (len(hits) > 0), hits

# retest the two sentences that just showed the bug
test_texts = [
    "The Republican president assumed he was helping the industry at the expense of the environment – a trade-off Trump was happy to make since he rejects climate science anyway.",
    "Victorina Morales, an undocumented immigrant from Guatemala, told reporters in a conference call on Tuesday that she was allowed to work at the resort after she submitted a fraudulent Social Security number and green card, which she alleges her supervisors knew were phony."
]
for t in test_texts:
    print(t)
    print(check_selective_emphasis(t))
    print('---')

The Republican president assumed he was helping the industry at the expense of the environment – a trade-off Trump was happy to make since he rejects climate science anyway.
(False, [])
---
Victorina Morales, an undocumented immigrant from Guatemala, told reporters in a conference call on Tuesday that she was allowed to work at the resort after she submitted a fraudulent Social Security number and green card, which she alleges her supervisors knew were phony.
(False, [])
---


In [24]:
selective_emphasis_signals = {
    "every", "all", "none", "only", "solely", "entirely", "completely",
    "the fact that", "underscores", "reaffirms", "proves that"
}

def check_selective_emphasis(text):
    text_lower = text.lower()
    hits = [w for w in selective_emphasis_signals if re.search(r'\b' + re.escape(w) + r'\b', text_lower)]
    hits += [p for p in selective_emphasis_phrases if p in text_lower]
    return (len(hits) > 0), hits

for t in [df_clean['text'].iloc[0], df_clean['text'].iloc[27]]:
    print(t)
    print(check_selective_emphasis(t))
    print('---')

The Republican president assumed he was helping the industry at the expense of the environment – a trade-off Trump was happy to make since he rejects climate science anyway. 
(False, [])
---
Victorina Morales, an undocumented immigrant from Guatemala, told reporters in a conference call on Tuesday that she was allowed to work at the resort after she submitted a fraudulent Social Security number and green card, which she alleges her supervisors knew were phony.
(False, [])
---


In [25]:
def assign_categories(text, lexicon):
    categories = {}

    rule_cats = classify_categories(text, lexicon)  # your existing 4-category function
    for cat_name, evidence in rule_cats:
        categories[cat_name] = evidence

    dehuman_flag, dehuman_evidence = check_dehumanising_glorifying(text)
    if dehuman_flag:
        categories['Dehumanising/Glorifying Framing'] = dehuman_evidence

    selective_flag, selective_evidence = check_selective_emphasis(text)
    if selective_flag:
        categories['Selective Emphasis'] = selective_evidence

    if not categories:
        categories['General Subjective Language'] = []

    return categories

# test against all 40 labeled sentences
for i, row in labeled.iterrows():
    result = assign_categories(row['text'], mpqa_lex)
    print(f"[{row['index']}]", list(result.keys()))

[0] ['Emotional Amplification']
[1] ['Emotional Amplification']
[2] ['Emotional Amplification']
[3] ['Emotional Amplification', 'Certainty Distortion']
[4] ['Emotional Amplification']
[5] ['Weasel Attribution']
[6] ['Emotional Amplification', 'Implicit Judgment', 'Selective Emphasis']
[7] ['General Subjective Language']
[8] ['Emotional Amplification', 'Selective Emphasis']
[9] ['Emotional Amplification', 'Weasel Attribution', 'Implicit Judgment']
[10] ['General Subjective Language']
[11] ['Dehumanising/Glorifying Framing']
[12] ['Emotional Amplification', 'Selective Emphasis']
[13] ['General Subjective Language']
[14] ['General Subjective Language']
[15] ['General Subjective Language']
[16] ['Implicit Judgment']
[17] ['General Subjective Language']
[18] ['Weasel Attribution']
[19] ['Emotional Amplification']
[20] ['Implicit Judgment']
[21] ['Emotional Amplification']
[22] ['Emotional Amplification']
[23] ['Emotional Amplification', 'Implicit Judgment']
[24] ['Emotional Amplification', 

In [26]:
hedge_words = {"may", "might", "could", "possibly", "reportedly", "allegedly", "seemingly", "apparently"}
certainty_words = {"definitely", "certainly", "undoubtedly", "clearly", "obviously", "always", "never"}
reporting_verbs = {"say", "said", "claim", "claimed", "argue", "argued", "believe", "believed"}

def classify_categories(text, lexicon):
    doc = nlp(text)
    words_lower = [t.text.lower() for t in doc]
    categories = []

    # 1. Emotional Amplification — now requires 2+ strongsubj hits, excluding proper nouns
    strong_hits = []
    for t in doc:
        if t.pos_ == "PROPN":
            continue
        entry = lexicon.get(t.text.lower(), (None, None))
        if entry[0] == 'strongsubj':
            strong_hits.append(t.text)
    strong_hits = list(dict.fromkeys(strong_hits))  # dedupe
    if len(strong_hits) >= 2:
        categories.append(('Emotional Amplification', strong_hits))

    # 2. Weasel Attribution — unchanged
    for tok in doc:
        if tok.lemma_.lower() in reporting_verbs:
            window_start = max(0, tok.i - 3)
            nearby_ents = [e for e in doc.ents if e.label_ in ("PERSON","ORG") and window_start <= e.start < tok.i]
            if not nearby_ents:
                categories.append(('Weasel Attribution', [tok.text]))
                break

    # 3. Certainty Distortion — unchanged
    hedges_found = [w for w in words_lower if w in hedge_words]
    certainty_found = [w for w in words_lower if w in certainty_words]
    if hedges_found or certainty_found:
        categories.append(('Certainty Distortion', hedges_found + certainty_found))

    # 4. Implicit Judgment — unchanged
    if any(tok.dep_ in ("nsubjpass", "auxpass") for tok in doc):
        categories.append(('Implicit Judgment', ['passive construction']))

    return categories

# rerun full assign_categories over all 40
for i, row in labeled.iterrows():
    result = assign_categories(row['text'], mpqa_lex)
    print(f"[{row['index']}]", list(result.keys()))

[0] ['Emotional Amplification']
[1] ['Emotional Amplification']
[2] ['Emotional Amplification']
[3] ['Emotional Amplification', 'Certainty Distortion']
[4] ['Emotional Amplification']
[5] ['Weasel Attribution']
[6] ['Emotional Amplification', 'Implicit Judgment', 'Selective Emphasis']
[7] ['General Subjective Language']
[8] ['Emotional Amplification', 'Selective Emphasis']
[9] ['Emotional Amplification', 'Weasel Attribution', 'Implicit Judgment']
[10] ['General Subjective Language']
[11] ['Dehumanising/Glorifying Framing']
[12] ['Emotional Amplification', 'Selective Emphasis']
[13] ['General Subjective Language']
[14] ['General Subjective Language']
[15] ['General Subjective Language']
[16] ['Implicit Judgment']
[17] ['General Subjective Language']
[18] ['Weasel Attribution']
[19] ['Emotional Amplification']
[20] ['Implicit Judgment']
[21] ['Emotional Amplification']
[22] ['Emotional Amplification']
[23] ['Emotional Amplification', 'Implicit Judgment']
[24] ['Emotional Amplification', 

In [31]:
def generate_explanation(category, evidence):
    templates = {
        'Emotional Amplification': f"This sentence uses strongly loaded words ({', '.join(evidence)}) instead of neutral language.",
        'Weasel Attribution': f"This sentence attributes a claim using '{evidence[0]}' without naming a specific source." if evidence else "This sentence attributes a claim without naming a specific source.",
        'Certainty Distortion': f"This sentence uses words like '{', '.join(evidence)}' that overstate or hedge certainty.",
        'Implicit Judgment': "This sentence uses passive voice, which can imply judgment while avoiding a direct claim.",
        'Selective Emphasis': f"This sentence uses absolute or emphatic framing ('{', '.join(evidence)}') that may overstate the case.",
        'Dehumanising/Glorifying Framing': f"This sentence uses framing language ({', '.join(evidence)}) that dehumanises or glorifies its subject.",
        'General Subjective Language': "This sentence was flagged as biased, but doesn't clearly match a specific pattern in our system.",
    }
    return templates.get(category, "This sentence was flagged as potentially biased.")

# quick test using real assign_categories output
for i, row in labeled.iloc[:5].iterrows():
    result = assign_categories(row['text'], mpqa_lex)
    print(row['text'])
    for cat, evidence in result.items():
        print(f"  [{cat}] -> {generate_explanation(cat, evidence)}")
    print('---')

For Fox, Clinton, Omar, and Ocasio-Cortez represent perfect targets in terms of riling up its intolerant base: Clinton is a woman of a certain age, while Ocasio-Cortez and Omar are women of a certain heritage.
  [Emotional Amplification] -> This sentence uses strongly loaded words (perfect, intolerant) instead of neutral language.
---
At the time, Moynihan was primarily concerned that irresponsible politicians promising their constituents an insupportable array of handouts would jeopardize America's economic well-being.
  [Emotional Amplification] -> This sentence uses strongly loaded words (concerned, irresponsible, promising, insupportable, jeopardize) instead of neutral language.
---
Trump proceeded to ignore multiple other attempts made over subsequent weeks by his advisers and Republican allies to get him to take the pandemic seriously. Instead he dismissed their warnings as "alarmist" and would go on to hold a number of rallies (seven) and golfing outings (three) between learning

In [32]:
!pip install feedparser trafilatura --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.8/248.8 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 97.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 23.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [33]:
import feedparser
import trafilatura

feed_url = "http://feeds.bbci.co.uk/news/world/rss.xml"
feed = feedparser.parse(feed_url)

print(f"{len(feed.entries)} entries found")
for i, entry in enumerate(feed.entries[:5]):
    print(f"[{i}] {entry.title}")
    print(f"    {entry.link}")

27 entries found
[0] Anthropic boss Dario Amodei calls for AI development to slow down
    https://www.bbc.co.uk/news/articles/c14dpgm0rg4o?at_medium=RSS&at_campaign=rss
[1] Saudi Arabia shuts key oil pipeline after drone attack launched from Iraq
    https://www.bbc.co.uk/news/articles/c62m933465eo?at_medium=RSS&at_campaign=rss
[2] They lost their jobs after posting about Charlie Kirk, but some have no regrets
    https://www.bbc.co.uk/news/articles/cj06jgl9qzlo?at_medium=RSS&at_campaign=rss
[3] Trump's comments on a united Ireland may have targeted audience across the Atlantic
    https://www.bbc.co.uk/news/articles/cwyzppd1d5lo?at_medium=RSS&at_campaign=rss
[4] Fire at nursing home in Chile kills 16 residents
    https://www.bbc.co.uk/news/articles/cy4zpp20w77o?at_medium=RSS&at_campaign=rss


In [34]:
chosen_index = 0  # change this after seeing the list above

article_url = feed.entries[chosen_index].link
downloaded = trafilatura.fetch_url(article_url)
article_text = trafilatura.extract(downloaded)

print("URL:", article_url)
print()
print(article_text[:2000])  # preview first ~2000 chars
print()
print("Total length:", len(article_text) if article_text else 0)

URL: https://www.bbc.co.uk/news/articles/c14dpgm0rg4o?at_medium=RSS&at_campaign=rss

Anthropic boss Dario Amodei calls for AI development to slow down
- Published
The head of AI company Anthropic has called for the pace of development of artificial intelligence models to slow down and to be closely monitored.
In an essay, Dario Amodei said developing AI was not in question, but the risks associated with it were "serious" and companies and governments must be given time to address them.
The bosses of two rival AI firms, Sam Altman of OpenAI and Elon Musk, have both said they agree with Amodei.
There have been growing concerns recently about the technology's potential risks. An AI researcher who left Anthropic this week told the BBC that "if we don't slow down at the current rate of progress, there is a strong chance that we could all die in the immediate future".
Jacob Coxon told Laura Kuenssberg that people working at AI companies were "genuinely frightened... genuinely concerned about

In [39]:
def extract_entities(text):
    doc = nlp(text)
    return {ent.text.lower().strip() for ent in doc.ents if ent.label_ in ("PERSON", "ORG", "GPE", "NORP")}

def check_summary_consistency(source_text, summary_text):
    source_entities = extract_entities(source_text)
    summary_entities = extract_entities(summary_text)
    
    unverified = []
    for ent in summary_entities:
        found = any(ent in src_ent or src_ent in ent for src_ent in source_entities) or (ent in source_text.lower())
        if not found:
            unverified.append(ent)
    
    return {
        'unverified_entities': unverified,
        'flag': len(unverified) > 0
    }

def generate_summary_with_safeguard(article_text, llm):
    summary = summarize_article(article_text, llm)
    consistency = check_summary_consistency(article_text, summary)
    
    disclaimer = "⚠️ Summaries are AI-generated and may contain errors. Please verify important facts against the original article."
    if consistency['flag']:
        disclaimer += f" Note: the following names/entities in this summary could not be verified against the source article: {', '.join(consistency['unverified_entities'])}."
    
    return {
        'summary': summary,
        'disclaimer': disclaimer,
        'flagged': consistency['flag'],
        'unverified_entities': consistency['unverified_entities']
    }

# run it end to end
result = generate_summary_with_safeguard(article_text, llm)
print("SUMMARY:")
print(result['summary'])
print()
print("DISCLAIMER:")
print(result['disclaimer'])
print()
print("Flagged:", result['flagged'])

SUMMARY:
Dario Amodei, the CEO of Anthropic, has called for a slower pace of AI development and monitoring. He believes that the risks associated with AI are "serious" and that companies and governments must be given time to address these risks. Amodei proposed a three-point plan that includes independent monitoring of AI models, industry-wide regulation, and global regulation. He also called for third-party monitors who could evaluate the safety of models as they are developed. The head of AI company Anthropic has called for the pace of development of artificial intelligence models to slow down.

DISCLAIMER:
⚠️ Summaries are AI-generated and may contain errors. Please verify important facts against the original article.

Flagged: False


In [37]:
print(article_text)

Anthropic boss Dario Amodei calls for AI development to slow down
- Published
The head of AI company Anthropic has called for the pace of development of artificial intelligence models to slow down and to be closely monitored.
In an essay, Dario Amodei said developing AI was not in question, but the risks associated with it were "serious" and companies and governments must be given time to address them.
The bosses of two rival AI firms, Sam Altman of OpenAI and Elon Musk, have both said they agree with Amodei.
There have been growing concerns recently about the technology's potential risks. An AI researcher who left Anthropic this week told the BBC that "if we don't slow down at the current rate of progress, there is a strong chance that we could all die in the immediate future".
Jacob Coxon told Laura Kuenssberg that people working at AI companies were "genuinely frightened... genuinely concerned about the fate of humanity in the next two years".
"It's not at all an exaggeration to say